# Model Interpretation for Image Data using Deep Learning Models

This notebook provides a comprehensive guide to applying model interpretability algorithms on pretrained deep learning models, specifically ResNet and VGG. It aims to help users understand how these models make predictions and to evaluate the importance of various input features in those predictions through visualizations.

## Objectives

- **Load and Prepare Models and Data**: Load pretrained ResNet and VGG models and prepare a sample image for testing.
- **Interpretability Techniques**:
  - **Integrated Gradients** (with and without Noise Tunnel): Attributes the prediction of a deep learning model to its input features by integrating gradients along a path from a baseline input to the actual input. The Noise Tunnel applies Gaussian noise to smooth attributions.
  - **Occlusion Sensitivity**: Evaluates feature importance by systematically occluding (masking) different parts of the input image and observing changes in the model’s output.
  - **Layer Grad-CAM**: Generates heatmaps that highlight the areas of the input image important for the model’s predictions, based on gradients flowing into the last convolutional layer.
  - **GradientShap**: Combines gradients with Shapley values to provide a more comprehensive attribution of the model’s predictions.
  - **Layer-wise Relevance Propagation (LRP)**: Analyzes the contribution of each pixel to the model’s output by propagating relevance scores backward through the network.

By the end of this notebook, you will be able to implement these techniques using the Captum library in PyTorch. Additionally, this guide will show how to overlay attributions on the input image, helping you visualize which parts of the image are significant in the model’s predictions.


In [ ]:
!pip install torch torchvision captum Flask-Compress
#!pip install matplotlib==3.4.3

## Model and Image Setup

In this step, we will load a pre-trained **Residual Network (ResNet)** model and set it to evaluation mode. Additionally, we will prepare the necessary libraries and define the image preprocessing steps required for the model.

### Load Required Libraries

We will begin by importing the necessary libraries, including PyTorch, torchvision, and Captum for model interpretability.

### Load Pre-trained ResNet Model

Next, we will load the ResNet18 model that has been pre-trained on the ImageNet dataset. Setting the model to evaluation mode is crucial as it disables dropout layers and batch normalization updates, which is essential when making predictions.




In [ ]:
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.models as models

from captum.attr import IntegratedGradients, Occlusion, LayerGradCam, LayerAttribution
from captum.attr import visualization as viz

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image
import requests

import torch
import torch.nn.functional as F

from PIL import Image

import os
import json
import numpy as np
from matplotlib.colors import LinearSegmentedColormap

import torchvision
from torchvision import models
from torchvision import transforms

from captum.attr import IntegratedGradients
from captum.attr import GradientShap
from captum.attr import LRP
from captum.attr import Occlusion
from captum.attr import NoiseTunnel
from captum.attr import visualization as viz
from captum.attr._utils.lrp_rules import EpsilonRule, GammaRule, Alpha1_Beta0_Rule

# Load a pre-trained ResNet model
model = models.resnet18(weights='IMAGENET1K_V1')
model = model.eval()  # Set model to evaluation mode

##  Load and Display Sample Image

In [ ]:
# Load and display the test image
test_img = Image.open('swan.jpg')
test_img_data = np.asarray(test_img)
plt.imshow(test_img_data)
plt.title("Original Image")
plt.axis('off')
plt.show()

## Image Preprocessing

Before feeding the image into the model, it needs to be preprocessed. This involves resizing, cropping, and normalizing the image. The following steps will be performed:

1. **Resize** the image to `224x224` pixels, which is the expected input size for the ResNet model.
2. **Center Crop** the image to ensure that the important features are retained.
3. **Convert** the image to a tensor format that can be processed by the model.
4. **Normalize** the pixel values using the mean and standard deviation values specific to the ImageNet dataset.


In [ ]:
# Define preprocessing transformations
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])

transform_normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Transform and normalize the image
transformed_img = transform(test_img)
input_img = transform_normalize(transformed_img)
input_img = input_img.unsqueeze(0)  # Add batch dimension

##  Load ImageNet Class Labels

To interpret the model's predictions, we need to load the class labels corresponding to the ImageNet dataset. These labels will allow us to translate the model's output indices into human-readable class names.

We will download the class labels from a specified URL and store them in a list for later use. The following code demonstrates how to perform this operation:


In [ ]:
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
response = requests.get(url)
idx_to_labels = response.text.splitlines() if response.ok else print("Error loading labels")

##  Model Prediction

After preprocessing the input image, we can now use the pretrained ResNet model to make predictions. The model will output scores for each class, which we will convert into probabilities using the softmax function. We will then identify the class with the highest score as the predicted label.


In [ ]:
output = model(input_img)
output = F.softmax(output, dim=1)
prediction_score, pred_label_idx = torch.topk(output, 1)
pred_label_idx = pred_label_idx.squeeze()
predicted_label = idx_to_labels[pred_label_idx.item()]
print('Predicted:', predicted_label, '(', prediction_score.item(), ')')

##  Gradient-based attribution

Integrated Gradients is a powerful method for attributing the predictions of deep learning models to their input features. This technique helps us understand which parts of the input image contributed most to the model's prediction.

In this step, we will initialize the Integrated Gradients method, compute the attributions for the input image, and visualize the results as a heatmap overlaying the original image.


In [ ]:
# Initialize Integrated Gradients
integrated_gradients = IntegratedGradients(model)

# Calculate attributions
attributions_ig = integrated_gradients.attribute(input_img, target=pred_label_idx, n_steps=200)

default_cmap = LinearSegmentedColormap.from_list('custom blue',
                                                 [(0, '#ffffff'),
                                                  (0.25, '#000000'),
                                                  (1, '#000000')], N=256)

_ = viz.visualize_image_attr(np.transpose(attributions_ig.squeeze().cpu().detach().numpy(), (1,2,0)),
                             np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1,2,0)),
                             method='heat_map',
                             cmap=default_cmap,
                             show_colorbar=True,
                             sign='positive',
                             outlier_perc=1)

## Attributions Using Integrated Gradients with Noise Tunnel

Lets compute attributions using Integrated Gradients and smooth them across multiple images generated by a Noise Tunnel. The Noise Tunnel adds Gaussian noise to the input with a standard deviation of one, generating 10 noisy samples (`nt_samples=10`). This technique helps in smoothing the attributions.

The smoothing process is performed using the `smoothgrad_sq` technique, which represents the mean of the squared attributions across the `nt_samples`. This method enhances the interpretability of the attributions by reducing the variance caused by noise.


In [ ]:
# Assuming 'input_img' is the tensor representing your input image
noise_tunnel = NoiseTunnel(integrated_gradients)

# Calculate attributions with the correct input tensor
attributions_ig_nt = noise_tunnel.attribute(input_img, nt_samples=10, nt_type='smoothgrad_sq', target=pred_label_idx)

# Visualize attributions
_ = viz.visualize_image_attr_multiple(
    np.transpose(attributions_ig_nt.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    ["original_image", "heat_map"],
    ["all", "positive"],
    cmap=default_cmap,
    show_colorbar=True
)


## GradientShap Attribution

**GradientShap** is a linear explanation model that utilizes a distribution of reference samples to explain the predictions made by the model. This method allows us to better understand which features of the input image contribute most significantly to the model's predictions.

#### Steps:

1. **Setting Seeds**: To ensure reproducibility, we set manual seeds for both PyTorch and NumPy.

2. **Initialize Gradient Shap**: We create an instance of the `GradientShap` class, passing in the pretrained model.

3. **Define Baseline Distribution**: We create a baseline distribution of images using the input image by concatenating two tensors: one with zeros (representing no contribution) and one with the input image itself.

4. **Calculate Attributions**: We compute the attributions for the input image using the `attribute` method. This method requires specifying the number of samples to be drawn (`n_samples`), the standard deviations for perturbation (`stdevs`), the baseline images, and the target class index for which the attributions are calculated.

5. **Visualize Attributions**: Finally, we visualize the calculated attributions using the `visualize_image_attr_multiple` function. This function overlays a heatmap of the attributions on the original image, helping us to identify the areas of significance.





In [ ]:
# Initialize GradientShap with the model
gradient_shap = GradientShap(model)

# Define baseline distribution of images using input_img
rand_img_dist = torch.cat([input_img * 0, input_img * 1])

# Calculate attributions
attributions_gs = gradient_shap.attribute(
    input_img,  # Make sure this is your input tensor
    n_samples=50,
    stdevs=0.0001,
    baselines=rand_img_dist,
    target=pred_label_idx
)

# Visualize attributions
_ = viz.visualize_image_attr_multiple(
    np.transpose(attributions_gs.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    ["original_image", "heat_map"],
    ["all", "absolute_value"],
    cmap=default_cmap,
    show_colorbar=True
)


## Occlusion-based Attribution

Now let us try a different approach to attribution. Occlusion-based attribution allows us to estimate which areas of the image are critical for the classifier's decision by systematically occluding parts of the image and observing how the decision changes.

In this method, we use a sliding window of size 15x15 (defined via `sliding_window_shapes`) with a stride of 8 along both dimensions of the image (defined via `strides`). At each position of the sliding window, we occlude the image with a baseline value of 0, which corresponds to a gray patch (defined via `baselines`).

**Note:** This computation might take more than one minute to complete, as the model is evaluated at every position of the sliding window.


In [ ]:
# Initialize Occlusion
occlusion = Occlusion(model)

# Calculate occlusion-based attributions
attributions_occ = occlusion.attribute(input_img, target=pred_label_idx, strides=(3, 8, 8),
                                       sliding_window_shapes=(3, 15, 15), baselines=0)

# Display the attributions with multiple visualization methods
_ = viz.visualize_image_attr_multiple(attributions_occ.squeeze().cpu().detach().numpy().transpose(1, 2, 0),
                                      transformed_img.permute(1,2,0).numpy(),
                                      ["original_image", "heat_map", "heat_map", "masked_image"],
                                      ["all", "positive", "negative", "positive"],
                                      show_colorbar=True,
                                      titles=["Original", "Positive Attribution", "Negative Attribution", "Masked"],
                                      fig_size=(18, 6))

To further verify the critical areas contributing to the model's prediction, particularly focusing on the upper part of the goose, especially the beak, we will occlude the image using a larger sliding window.

By using a larger sliding window, we can assess how occluding larger sections of the image affects the model's predictions. This can help us understand the significance of broader features in the classification process.


In [ ]:
# Initialize Occlusion with the model
occlusion = Occlusion(model)

# Calculate occlusion-based attributions
attributions_occ = occlusion.attribute(input_img,  # Use 'input_img' instead of 'input'
                                       strides=(3, 50, 50),
                                       target=pred_label_idx,
                                       sliding_window_shapes=(3, 60, 60),
                                       baselines=0)

# Visualize the attributions
_ = viz.visualize_image_attr_multiple(
    np.transpose(attributions_occ.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    ["original_image", "heat_map"],
    ["all", "positive"],
    show_colorbar=True,
    outlier_perc=2
)


##  Layer Grad-CAM Attribution

Layer Grad-CAM (Gradient-weighted Class Activation Mapping) is a technique that visualizes the important regions of an input image that contribute to the model's predictions. It works by using the gradients flowing into a specified convolutional layer to generate a heatmap that highlights relevant features in the input image.

In this step, we will initialize the Layer Grad-CAM method for a specific convolutional layer, calculate the Grad-CAM attributions, and visualize the results.


In [ ]:
# Initialize Layer Grad-CAM for a specific layer
layer_gradcam = LayerGradCam(model, model.layer3[1].conv2)

# Calculate Grad-CAM attributions
attributions_lgc = layer_gradcam.attribute(input_img, target=pred_label_idx)

# Display Grad-CAM attributions
_ = viz.visualize_image_attr(attributions_lgc[0].cpu().permute(1, 2, 0).detach().numpy(), sign="all", title="Layer Grad-CAM Attribution")

## Layer-wise Relevance Propagation (LRP) based Attribution

Now let's explore a different approach called **Layer-Wise Relevance Propagation (LRP)**. LRP employs a backward propagation mechanism that is applied sequentially to all layers of the model to identify which neurons contributed to the output. The output score from LRP represents the relevance of each input feature, decomposed into values for each layer. This decomposition is defined by rules that may vary for each layer.

In this implementation, we will use the default **Epsilon-Rule** for each layer, which is the standard approach for calculating LRP attributions. This rule allows us to understand how each part of the input influences the final prediction.

**Note:** We are using the VGG16 model in this example, as the default rules for LRP are not currently fine-tuned for ResNet, making VGG16 a more suitable choice for demonstrating this method.


In [ ]:
# Load the pre-trained VGG16 model and set it to evaluation mode
model = models.vgg16(pretrained=True)
model.eval()

# Initialize LRP with the model
lrp = LRP(model)

# Calculate LRP attributions
attributions_lrp = lrp.attribute(input_img,  # Ensure 'input_img' is used here
                                 target=pred_label_idx)

# Visualize the attributions
_ = viz.visualize_image_attr_multiple(
    np.transpose(attributions_lrp.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    ["original_image", "heat_map"],
    ["all", "positive"],
    show_colorbar=True,
    outlier_perc=2
)

## Layer-wise Relevance Propagation (LRP) with Custom Rules

Lets utilize Layer-wise Relevance Propagation (LRP) to analyze which neurons in the VGG16 model contributed to the output. LRP works by propagating the output relevance backwards through the network, layer by layer, to assign relevance scores to each input feature.

#### Steps:

1. **Get the Layers of the Model**: We extract the layers from the VGG16 model, combining both the feature and classifier parts. This allows us to modify the relevance propagation rules for different segments of the model.

2. **Assign LRP Rules**: We assign different LRP rules to the layers based on their indices:
   - For the first 16 layers (usually convolutional layers), we apply the **GammaRule**.
   - For layers 17 to 30, we use the **EpsilonRule**.
   - For layers 31 and onward, we use another **EpsilonRule** with `epsilon=0` to minimize its effect.

3. **Initialize LRP**: We create an instance of the `LRP` class with the modified model to calculate attributions based on the defined rules.

4. **Calculate LRP Attributions**: We compute the LRP attributions for the input image using the `attribute` method, specifying the target class index.

5. **Visualize the Attributions**: Finally, we visualize the computed attributions alongside the original image, displaying a heatmap that highlights the significant regions contributing to the prediction.


In [ ]:
# Get the layers of the model
layers = list(model._modules["features"]) + list(model._modules["classifier"])
num_layers = len(layers)

# Assign LRP rules to the layers
for idx_layer in range(1, num_layers):
    if idx_layer <= 16:
        setattr(layers[idx_layer], "rule", GammaRule())
    elif 17 <= idx_layer <= 30:
        setattr(layers[idx_layer], "rule", EpsilonRule())
    elif idx_layer >= 31:
        setattr(layers[idx_layer], "rule", EpsilonRule(epsilon=0))

# Initialize LRP with the modified model
lrp = LRP(model)

# Calculate LRP attributions
attributions_lrp = lrp.attribute(input_img,  # Ensure to use 'input_img'
                                  target=pred_label_idx)

# Visualize the attributions
_ = viz.visualize_image_attr_multiple(
    np.transpose(attributions_lrp.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    np.transpose(transformed_img.squeeze().cpu().detach().numpy(), (1, 2, 0)),
    ["original_image", "heat_map"],
    ["all", "positive"],
    show_colorbar=True,
    outlier_perc=2
)
